In [1]:
from pathlib import Path
import pandas as pd

# Locate the dataset relative to the notebooks folder.
data_path = Path("../data/raw/diabetic_data.csv")

# Stop with a clear message if the file is missing.
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

# Treat "?" as missing while preserving literal "None" categories.
df = pd.read_csv(
    data_path,
    keep_default_na=False,
    na_values=["?"]
)

# Check the dataset dimensions.
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 101766
Columns: 50


/var/folders/lw/dbrf54n571n8b9g__cjb5_6h0000gn/T/ipykernel_99820/2685397114.py:12: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
# Create a working copy and remove weight due to extensive missing data.
model_df = df.drop(columns=["weight"]).copy()

# Convert the original readmission labels into a binary target.
model_df["readmitted_30_days"] = model_df["readmitted"].map(
    {"<30": 1, ">30": 0, "NO": 0}
)

# Check that every original label was mapped successfully.
if model_df["readmitted_30_days"].isna().any():
    raise ValueError("Unexpected or missing readmission labels.")

# Count encounters in each target class.
print(model_df["readmitted_30_days"].value_counts().sort_index())

readmitted_30_days
0    90409
1    11357
Name: count, dtype: int64


In [3]:
# 1. Store the answer the model will learn to predict.
y = model_df["readmitted_30_days"].copy()

# 2. Keep patient IDs separately for the train/test split.
# All encounters from the same patient must stay in one set.
patient_groups = model_df["patient_nbr"].copy()

# 3. Remove the answer, its original source, and identifiers from inputs.
X = model_df.drop(columns=[
    "readmitted_30_days",
    "readmitted",
    "encounter_id",
    "patient_nbr"
]).copy()

# 4. Check that inputs and target have the same number of rows.
print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Unique patients:", patient_groups.nunique())

Input shape: (101766, 46)
Target shape: (101766,)
Unique patients: 71518


In [4]:
from sklearn.model_selection import GroupShuffleSplit

# 1. Reserve 20% of patients for testing.
# A fixed seed makes the split reproducible.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# 2. Split by patient ID rather than individual encounters.
train_idx, test_idx = next(
    splitter.split(X, y, groups=patient_groups)
)

# 3. Select the corresponding inputs and answers.
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# 4. Verify that no patient appears in both sets.
train_patients = set(patient_groups.iloc[train_idx])
test_patients = set(patient_groups.iloc[test_idx])
overlap = train_patients.intersection(test_patients)

assert len(overlap) == 0, "Patient overlap detected."

print("Training encounters:", len(X_train))
print("Test encounters:", len(X_test))
print("Patients in both sets:", len(overlap))

Training encounters: 81613
Test encounters: 20153
Patients in both sets: 0


In [5]:
# Check the readmission proportion in each set.
split_summary = pd.DataFrame({
    "encounters": [len(y_train), len(y_test)],
    "early_readmissions": [y_train.sum(), y_test.sum()],
    "readmission_rate_pct": [
        y_train.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["train", "test"])

split_summary.round(2)

,encounters,early_readmissions,readmission_rate_pct
train,81613,9206,11.28
test,20153,2151,10.67


In [6]:
# Inspect training features before choosing preprocessing steps.
feature_summary = pd.DataFrame({
    "data_type": X_train.dtypes.astype(str),
    "missing_pct": X_train.isna().mean() * 100,
    "unique_values": X_train.nunique()
})

# Show features with the highest missing percentages first.
feature_summary.sort_values(
    "missing_pct", ascending=False
).round(2)

,data_type,missing_pct,unique_values
medical_specialty,str,48.81,71
payer_code,str,39.60,16
race,str,2.28,5
diag_3,str,1.39,754
diag_2,str,0.36,719
diag_1,str,0.02,700
troglitazone,str,0.00,2
glyburide,str,0.00,4
tolbutamide,str,0.00,2
pioglitazone,str,0.00,4


In [7]:
# 1. Find features with only one distinct value in training.
# Include missing values when counting distinct values.
constant_columns = X_train.columns[
    X_train.nunique(dropna=False) <= 1
].tolist()

# 2. Remove the same columns from training and test inputs.
X_train = X_train.drop(columns=constant_columns)
X_test = X_test.drop(columns=constant_columns)

# 3. Show which features were removed and the remaining dimensions.
print("Removed constant features:", constant_columns)
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Removed constant features: ['acetohexamide', 'examide', 'citoglipton', 'metformin-pioglitazone']
Training shape: (81613, 42)
Test shape: (20153, 42)


In [8]:
# 1. List features that represent durations or counts.
numeric_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# 2. Treat the remaining features as categories.
# This includes admission codes, age groups, and diagnosis codes.
categorical_columns = [
    column for column in X_train.columns
    if column not in numeric_columns
]

# 3. Convert categories to object dtype for later preprocessing.
# Apply the same conversion to training and test data.
X_train[categorical_columns] = (
    X_train[categorical_columns].astype("object")
)
X_test[categorical_columns] = (
    X_test[categorical_columns].astype("object")
)

# 4. Verify the number of features in each group.
print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

Numeric features: 8
Categorical features: 34


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Define preprocessing for durations and counts.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 2. Define preprocessing for categorical features.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 3. Apply each pipeline to its corresponding columns.
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Preprocessing steps defined.")

Preprocessing steps defined.


In [10]:
# 1. Learn medians, scales, and categories from training data only.
X_train_prepared = preprocessor.fit_transform(X_train)

# 2. Check the dimensions after encoding categorical features.
print("Before preprocessing:", X_train.shape)
print("After preprocessing:", X_train_prepared.shape)

Before preprocessing: (81613, 42)
After preprocessing: (81613, 2419)


In [11]:
# Apply preprocessing rules already learned from training.
# Do not use fit or fit_transform on test data.
X_test_prepared = preprocessor.transform(X_test)

# Both sets must have the same number and order of features.
assert X_train_prepared.shape[1] == X_test_prepared.shape[1]

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)

Prepared training shape: (81613, 2419)
Prepared test shape: (20153, 2419)


In [12]:
import numpy as np

# 1. Predict no early readmission for every test encounter.
baseline_predictions = np.zeros(len(y_test), dtype=int)

# 2. Calculate the proportion of correct predictions.
baseline_accuracy = (baseline_predictions == y_test.to_numpy()).mean()

# 3. Count actual early readmissions and those detected.
actual_readmissions = int(y_test.sum())
detected_readmissions = int(
    ((baseline_predictions == 1) & (y_test.to_numpy() == 1)).sum()
)

print(f"Baseline accuracy: {baseline_accuracy:.2%}")
print("Actual early readmissions:", actual_readmissions)
print("Detected early readmissions:", detected_readmissions)

Baseline accuracy: 89.33%
Actual early readmissions: 2151
Detected early readmissions: 0


In [13]:
from sklearn.linear_model import LogisticRegression

# 1. Define the first classification model.
logistic_model = LogisticRegression(
    class_weight="balanced",  # Weight classes by their training frequency.
    solver="liblinear",       # Optimization algorithm for this binary model.
    max_iter=1000,            # Maximum optimization iterations.
    random_state=42           # Make the randomized steps reproducible.
)

# 2. Learn from prepared training inputs and their actual answers.
logistic_model.fit(X_train_prepared, y_train)

print("Logistic regression training completed.")

Logistic regression training completed.


In [14]:
# 1. Predict the class for each test encounter.
logistic_predictions = logistic_model.predict(X_test_prepared)

# 2. Compare predictions with the actual answers.
actual = y_test.to_numpy()

true_positives = int(((actual == 1) & (logistic_predictions == 1)).sum())
false_negatives = int(((actual == 1) & (logistic_predictions == 0)).sum())
false_positives = int(((actual == 0) & (logistic_predictions == 1)).sum())
true_negatives = int(((actual == 0) & (logistic_predictions == 0)).sum())

# 3. Display the four possible outcomes.
print("Detected readmissions:", true_positives)
print("Missed readmissions:", false_negatives)
print("False alarms:", false_positives)
print("Correctly predicted no early readmission:", true_negatives)

Detected readmissions: 1197
Missed readmissions: 954
False alarms: 5946
Correctly predicted no early readmission: 12056


In [15]:
# Overall proportion of correct predictions.
accuracy = (
    (true_positives + true_negatives) / len(y_test)
)

# Proportion of actual readmissions that the model detected.
recall = (
    true_positives / (true_positives + false_negatives)
)

# Proportion of readmission alerts that were correct.
precision = (
    true_positives / (true_positives + false_positives)
)

print(f"Accuracy: {accuracy:.2%}")
print(f"Recall: {recall:.2%}")
print(f"Precision: {precision:.2%}")

Accuracy: 65.76%
Recall: 55.65%
Precision: 16.76%
